# Algorithm 5.1 — Gibbs' method of orbit determination

**Goal:** the first *inverse* problem. Given **three position vectors** $\mathbf{R}_1,\mathbf{R}_2,\mathbf{R}_3$ of a satellite (three sightings, no velocities, no times), recover the velocity $\mathbf{V}_2$ at the middle one — which, with $\mathbf{R}_2$, gives the full state and hence the orbit.

**Code (answer key):** [`gibbs`](../curtis/algorithms/alg_5_1_gibbs.py) · **Book:** §5.2, Algorithm 5.1 · **Worked example:** 5.1

It is pure vector algebra — no time of flight, no Kepler iteration. The catch: the three positions must be **coplanar** (genuinely on one orbit).

## Read first

| Symbol | Meaning |
|---|---|
| $\mathbf{R}_1,\mathbf{R}_2,\mathbf{R}_3$ | three position vectors, in time order (km) |
| $\mathbf{V}_2$ | the unknown: velocity at $\mathbf{R}_2$ (km/s) |
| $\mathbf{c}_{12}=\mathbf{R}_1\times\mathbf{R}_2$, etc. | pairwise cross products |
| $\mathbf{N},\mathbf{D},\mathbf{S}$ | Gibbs' constructed vectors (Eqs 5.13, 5.14, 5.21) |
| `ierr` | flag: $1$ if the three are not coplanar |

**The result (Eq 5.22):** $\;\mathbf{V}_2=\sqrt{\dfrac{\mu}{N\,D}}\left(\dfrac{\mathbf{D}\times\mathbf{R}_2}{r_2}+\mathbf{S}\right)$.

## The picture (three dots, one orbit)

A radar or telescope can give you *where* a satellite is, but not how fast it is moving. Gibbs' insight: three positions on the same orbit already contain the velocity — the orbit's geometry links them.

Two facts make it work:

1. **Coplanarity.** All orbital motion lies in one plane, so $\mathbf{R}_1,\mathbf{R}_2,\mathbf{R}_3$ must be coplanar: $\mathbf{R}_1\cdot(\mathbf{R}_2\times\mathbf{R}_3)=0$. If not (measurement error or different orbits), the method flags `ierr=1`.
2. **The orbit equation.** Each $\mathbf{R}_i$ satisfies $r_i=\dfrac{h^2/\mu}{1+e\cos\theta_i}$. Three such relations are enough to pin down the orbit and back out $\mathbf{V}_2$ — algebraically, with no iteration.

Once you have $(\mathbf{R}_2,\mathbf{V}_2)$, Algorithm 4.1 (`coe_from_sv`) gives the orbital elements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
from curtis.algorithms.alg_4_1_coe_from_sv import coe_from_sv          # to read off elements
from curtis.algorithms.alg_4_2_sv_from_coe import sv_from_coe          # to draw the orbit
from curtis.algorithms.alg_5_1_gibbs import gibbs as _gibbs_ref        # reference, for the picture only

In [ ]:
# Example 5.1: three sightings -> recover the orbit, drawn in its own plane.
mu = 398600.0
R1 = np.array([-294.32, 4265.1, 5986.7])
R2 = np.array([-1365.4, 3637.6, 6346.8])
R3 = np.array([-2940.3, 2473.7, 6555.8])

V2, _ = _gibbs_ref(R1, R2, R3, mu)                      # reference solve (drawing only)
h, e, RA, incl, w, TA, a = coe_from_sv(R2, V2, mu)

# Rotation ECI -> perifocal, so we can draw the orbit flat in its own plane.
def _R3(t): c,s=np.cos(t),np.sin(t); return np.array([[c,s,0],[-s,c,0],[0,0,1]])
def _R1(t): c,s=np.cos(t),np.sin(t); return np.array([[1,0,0],[0,c,s],[0,-s,c]])
Qe2p = _R3(w) @ _R1(incl) @ _R3(RA)
proj = lambda V: (Qe2p @ V)[:2]

th = np.linspace(0, 2*np.pi, 400)
p_sl = h**2/mu
ox, oy = (p_sl/(1+e*np.cos(th)))*np.cos(th), (p_sl/(1+e*np.cos(th)))*np.sin(th)
p1, p2, p3 = proj(R1), proj(R2), proj(R3)
v2p = proj(V2)

fig, ax = plt.subplots(figsize=(6.6, 6.2))
ax.plot(ox, oy, color='#73726c', lw=1.6)                       # the recovered orbit
ax.plot(0, 0, 'o', color='#3B8BD4', ms=9)                       # Earth (focus)
for p, lbl, col in [(p1,'R1','#888780'), (p2,'R2','#1D9E75'), (p3,'R3','#888780')]:
    ax.plot([0,p[0]],[0,p[1]], color=col, lw=1)
    ax.plot(*p, 'o', color=col, ms=6)
    ax.annotate(lbl, p, textcoords='offset points', xytext=(7,5), color=col)
ax.quiver(p2[0], p2[1], v2p[0], v2p[1], color='#D4537E', angles='xy',
          scale_units='xy', scale=0.006, width=0.006)
ax.annotate('recovered V2', p2 + 250*v2p/np.linalg.norm(v2p),
            color='#D4537E', textcoords='offset points', xytext=(6,-2))
ax.annotate('Earth', (0,0), textcoords='offset points', xytext=(8,-12), color='#3B8BD4')
ax.set_aspect('equal'); ax.axis('off')
ax.set_title("Gibbs: three positions -> the orbit through them (Example 5.1)")
plt.show()

## See it — the velocity falls out of the geometry

Drawn in the orbit's own plane, the three position tips $\mathbf{R}_1,\mathbf{R}_2,\mathbf{R}_3$ sit on one ellipse, and Gibbs hands back the velocity $\mathbf{V}_2$ (pink) tangent to the orbit at $\mathbf{R}_2$ — derived from the three positions alone, no clock involved.

That is the whole appeal: with three clean position fixes you get a complete state vector, and from there the orbital elements. The only thing that can go wrong is if the three fixes are **not coplanar** — then they cannot lie on a single Keplerian orbit, and `ierr` warns you.

## Where these equations come from

The full derivation (Curtis §5.2) is about a page of vector algebra; here is the skeleton of *why* each vector appears.

### Coplanarity first
A single orbit lives in one plane, so $\mathbf{R}_2\times\mathbf{R}_3$, $\mathbf{R}_3\times\mathbf{R}_1$, $\mathbf{R}_1\times\mathbf{R}_2$ are all parallel to $\mathbf{h}$. Equivalently $\mathbf{R}_1\cdot(\mathbf{R}_2\times\mathbf{R}_3)=0$ — the test the code makes (scaled, against a tolerance).

### Building $\mathbf{N}$, $\mathbf{D}$, $\mathbf{S}$
Writing each position with the orbit equation and eliminating the per-point unknowns, Gibbs found three combinations that collapse the algebra:
$$\mathbf{N}=r_1\,\mathbf{c}_{23}+r_2\,\mathbf{c}_{31}+r_3\,\mathbf{c}_{12}\ (5.13),\quad \mathbf{D}=\mathbf{c}_{12}+\mathbf{c}_{23}+\mathbf{c}_{31}\ (5.14),$$
$$\mathbf{S}=\mathbf{R}_1(r_2-r_3)+\mathbf{R}_2(r_3-r_1)+\mathbf{R}_3(r_1-r_2)\ (5.21).$$
Here $\mathbf{N}$ and $\mathbf{D}$ are both parallel to $\mathbf{h}$, and their magnitudes give the semi-latus rectum: $p=N/D$. $\mathbf{S}$ lies in the orbit plane and carries the eccentricity direction.

### The velocity
Combining $h=\sqrt{\mu p}$ with the perifocal velocity expression yields the closed form
$$\mathbf{V}_2=\sqrt{\frac{\mu}{N\,D}}\left(\frac{\mathbf{D}\times\mathbf{R}_2}{r_2}+\mathbf{S}\right)\qquad(5.22),$$
where $N=|\mathbf{N}|$, $D=|\mathbf{D}|$. No iteration — three positions in, one velocity out.

## Step by step (in code order)

1. **Magnitudes:** $r_1,r_2,r_3$.
2. **Cross products:** $\mathbf{c}_{12}=\mathbf{R}_1\times\mathbf{R}_2$, $\mathbf{c}_{23}=\mathbf{R}_2\times\mathbf{R}_3$, $\mathbf{c}_{31}=\mathbf{R}_3\times\mathbf{R}_1$.
3. **Coplanarity check:** if $\left|\dfrac{\mathbf{R}_1\cdot\mathbf{c}_{23}}{r_1\,|\mathbf{c}_{23}|}\right|>\text{tol}$, set `ierr=1`.
4. $\mathbf{N}=r_1\mathbf{c}_{23}+r_2\mathbf{c}_{31}+r_3\mathbf{c}_{12}$, $\;\mathbf{D}=\mathbf{c}_{12}+\mathbf{c}_{23}+\mathbf{c}_{31}$.
5. $\mathbf{S}=\mathbf{R}_1(r_2-r_3)+\mathbf{R}_2(r_3-r_1)+\mathbf{R}_3(r_1-r_2)$.
6. $\mathbf{V}_2=\sqrt{\mu/(N D)}\,(\mathbf{D}\times\mathbf{R}_2/r_2+\mathbf{S})$, with $N=|\mathbf{N}|,\,D=|\mathbf{D}|$.

**↓ Now type your implementation below.** All vector algebra (`np.cross`, `np.dot`, `np.linalg.norm`) — no loops. Return both `V2` and the `ierr` flag. Answer key linked above; peek only after you try.

In [ ]:
def gibbs(R1, R2, R3, mu, tol=1.0e-4):
    """Velocity V2 at R2 from three coplanar position vectors. Returns (V2, ierr)."""
    R1 = np.asarray(R1, float); R2 = np.asarray(R2, float); R3 = np.asarray(R3, float)
    ierr = 0

    # 1. Magnitudes r1, r2, r3

    # 2. Cross products  c12 = R1 x R2,  c23 = R2 x R3,  c31 = R3 x R1

    # 3. Coplanarity check: if |R1 . c23| / (r1 * |c23|) > tol:  ierr = 1

    # 4. N = r1*c23 + r2*c31 + r3*c12        (Eq 5.13)
    #    D = c12 + c23 + c31                  (Eq 5.14)

    # 5. S = R1*(r2-r3) + R2*(r3-r1) + R3*(r1-r2)   (Eq 5.21)

    # 6. V2 = sqrt(mu/(|N|*|D|)) * (cross(D, R2)/r2 + S)   (Eq 5.22)

    # return V2, ierr
    raise NotImplementedError("fill in steps 1-6, then delete this line")

## Verify — Example 5.1

**Input** ($\mu_\oplus=398600$):
$\mathbf{R}_1=[-294.32, 4265.1, 5986.7]$, $\mathbf{R}_2=[-1365.4, 3637.6, 6346.8]$, $\mathbf{R}_3=[-2940.3, 2473.7, 6555.8]$ km.

**Expected:** $\mathbf{V}_2=[-6.2176, -4.01237, 1.59915]$ km/s, and the orbit comes out as
$h=56193$, $e=0.100159$, $i=60.001°$, $\Omega=40.0023°$, $\omega=30.1093°$, $\theta=49.8894°$, $a=8002.14$ km.

Run once your function is typed.

In [ ]:
mu = 398600.0
R1 = np.array([-294.32, 4265.1, 5986.7])
R2 = np.array([-1365.4, 3637.6, 6346.8])
R3 = np.array([-2940.3, 2473.7, 6555.8])

V2, ierr = gibbs(R1, R2, R3, mu)
print(f"ierr = {ierr}   (0 = coplanar, good)")
print(f"V2 (km/s) = [{V2[0]:.6g} {V2[1]:.6g} {V2[2]:.6g}]   (expect [-6.2176  -4.01237  1.59915])")

assert ierr == 0
assert np.allclose(V2, [-6.2176, -4.01237, 1.59915], atol=1e-4)

# Turn the recovered state into orbital elements (Algorithm 4.1)
deg = np.pi/180
h, e, RA, incl, w, TA, a = coe_from_sv(R2, V2, mu)
print(f"\nh={h:.6g}  e={e:.6g}  i={incl/deg:.5g}  RA={RA/deg:.5g}  w={w/deg:.5g}  TA={TA/deg:.5g}  a={a:.6g}")
assert abs(e - 0.100159) < 1e-5
assert abs(incl/deg - 60.001) < 1e-2
assert abs(a - 8002.14) < 1e-1
print("\nAll checks passed ✔")

## What this confirms

- **Position alone carries velocity.** Three coplanar fixes determine a unique orbit; Gibbs extracts $\mathbf{V}_2$ from them with closed-form vector algebra — no time, no iteration.
- The **coplanarity test** is physical: three points that are not coplanar cannot share one Keplerian orbit, and `ierr` catches that.
- Chaining with `coe_from_sv` (4.1) takes you from raw position fixes all the way to orbital elements — a complete preliminary orbit determination.

**Next:** Lambert's problem (Algorithm 5.2) — the other classic OD method: given **two** positions and the **time of flight** between them, find the orbit. It leans directly on the universal-variable and Stumpff machinery you built in Chapter 3.